# 🌱 AI-Driven Crop Recommendation for Climate-Resilient & Sustainable Agriculture
---
| | |
|---|---|
| **Course** | Sustainability, Climate Actions & Environmental Sciences (MCC471) |
| **Topic Area** | D. Climate Change & Environmental Data Analytics |
| **Dataset** | IEEE DataPort – Crop Recommendation (18,079 samples) |
| **Features** | Nitrogen · Phosphorus · Potassium · pH · Rainfall · Temperature |
| **Target** | 40 Crop Classes |
| **Models** | Decision Tree · Random Forest · KNN · SVM · Naive Bayes |
| **XAI** | SHAP (tree-path method) · LIME (batch, full test set) |
| **SDGs Addressed** | SDG 2 · SDG 6 · SDG 12 · SDG 13 · SDG 15 |
| **Source** | https://ieee-dataport.org/documents/crop-recommendation-dataset |

---
### 🌍 Project Context

Climate change is reshaping agricultural ecosystems globally — shifting rainfall patterns, rising temperatures, and increasing soil stress are making traditional crop selection unreliable. This project applies **Machine Learning and Explainable AI (XAI)** to analyse environmental and soil data, recommending optimal crops that are both **climate-adapted and resource-efficient**.

By treating Rainfall and Temperature as first-class model features alongside soil nutrients, this project directly operationalises **SDG 13 (Climate Action)** and **SDG 2 (Zero Hunger)** — demonstrating that data-driven precision agriculture is a viable pathway toward sustainable food systems.

---
## 🎯 SDG Alignment

| SDG | Goal Name | Relevance to This Project |
|-----|-----------|--------------------------|
| **SDG 2** | Zero Hunger | Reduces crop failure by recommending climate-suited crops; improves food security |
| **SDG 6** | Clean Water & Sanitation | Rainfall-aware recommendations prevent water-intensive crops in low-rainfall zones |
| **SDG 12** | Responsible Consumption & Production | Prevents over-application of N, P, K fertilizers; reduces chemical runoff |
| **SDG 13** | Climate Action | Temperature & Rainfall are primary model drivers — directly quantifies climate's role in farming |
| **SDG 15** | Life on Land | Protects soil health by matching crop nutrient demands to actual soil composition |

---

## 📦 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

np.random.seed(42)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

# ── Feature configuration ──────────────────────────────────────────────────
FEATURES = ['N', 'P', 'K', 'pH', 'rainfall', 'temperature']
FLABELS  = ['Nitrogen (N)', 'Phosphorus (P)', 'Potassium (K)',
             'pH', 'Rainfall (mm)', 'Temperature (°C)']

# Group features by type for sustainability framing
SOIL_FEATURES    = ['N', 'P', 'K', 'pH']           # SDG 12, SDG 15
CLIMATE_FEATURES = ['rainfall', 'temperature']       # SDG 13, SDG 6

COLORS   = ['#3498DB','#2ECC71','#E74C3C','#9B59B6','#F39C12','#1ABC9C']

# SDG colour palette (used in sustainability visualisations)
SDG_COLORS = {
    'SDG 2':  '#DDA63A',   # Zero Hunger — gold
    'SDG 6':  '#26BDE2',   # Clean Water — blue
    'SDG 12': '#BF8B2E',   # Responsible Consumption — brown
    'SDG 13': '#3F7E44',   # Climate Action — green
    'SDG 15': '#56C02B',   # Life on Land — light green
}

print('✅  All libraries imported successfully!')
print(f'\nSoil features    (SDG 12 / SDG 15): {SOIL_FEATURES}')
print(f'Climate features (SDG 13 / SDG 6) : {CLIMATE_FEATURES}')

---
## 📂 2. Load Dataset

> **Dataset:** IEEE DataPort Crop Recommendation Dataset  
> 18,079 labelled samples with 6 environmental/soil features spanning 40 crop classes across diverse agro-climatic zones of India.
>
> Each sample represents a field observation with:
> - **Soil nutrients** → N, P, K (mg/kg), pH — indicators of soil health (SDG 15)
> - **Climate variables** → Rainfall (mm), Temperature (°C) — climate adaptation inputs (SDG 13)

In [ ]:
# NOTE: only Train_Dataset.csv is needed — we split it 80/20 internally
df = pd.read_csv('Train_Dataset.csv')
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

print(f'Dataset shape      : {df.shape}')
print(f'Crop classes       : {df["Crop"].nunique()}')
print(f'Missing values     : {df.isnull().sum().sum()}')
print(f'\nClimate feature ranges (SDG 13):')
print(f'  Rainfall    : {df["rainfall"].min():.1f} – {df["rainfall"].max():.1f} mm')
print(f'  Temperature : {df["temperature"].min():.1f} – {df["temperature"].max():.1f} °C')
print(f'\nSoil nutrient ranges (SDG 15):')
for feat in ['N','P','K','pH']:
    print(f'  {feat:<4s}: {df[feat].min():.2f} – {df[feat].max():.2f}')
print(f'\nCrop list: {sorted(df["Crop"].unique())}')
df.head(8)

---
## 📊 3. Exploratory Data Analysis
### 3a. Statistical Summary

> Understanding the range and distribution of climate and soil variables is the first step in assessing **how much climate variability** the recommendation system must handle — a key consideration for SDG 13.

In [ ]:
display(df[FEATURES].describe().round(3))
print('\nData types:')
print(df.dtypes)
print('\n📌 Sustainability note:')
print(f'  Rainfall CV (coefficient of variation): {df["rainfall"].std()/df["rainfall"].mean()*100:.1f}% — high climate variability')
print(f'  Temperature CV: {df["temperature"].std()/df["temperature"].mean()*100:.1f}%')
print(f'  → These ranges represent diverse agro-climatic zones; a one-size-fits-all')
print(f'    crop strategy would be environmentally and economically inefficient.')

### 3b. Feature Distributions — Soil & Climate Factors

> Features are colour-coded by sustainability dimension:
> - **Blue/Green/Red/Purple** → Soil nutrients (SDG 12 / SDG 15)
> - **Orange/Teal** → Climate variables (SDG 13 / SDG 6)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Environmental & Soil Feature Distributions\n'             'Inputs to the Climate-Adaptive Crop Recommendation System',
             fontsize=15, fontweight='bold')

sdg_tags = ['SDG 12/15', 'SDG 12/15', 'SDG 12/15', 'SDG 15', 'SDG 13/6', 'SDG 13']
for ax, feat, lbl, col, tag in zip(axes.flatten(), FEATURES, FLABELS, COLORS, sdg_tags):
    ax.hist(df[feat], bins=40, color=col, edgecolor='white', alpha=0.85)
    ax.set_title(f'{lbl}\n[{tag}]', fontweight='bold', fontsize=10)
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
    ax.axvline(df[feat].mean(), color='black', ls='--', lw=1.5,
               label=f'Mean={df[feat].mean():.1f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3c. Crop Class Distribution — 40 Agro-Climatic Crop Types

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
crop_counts = df['Crop'].value_counts().sort_values(ascending=False)
bar_colors  = plt.cm.tab20(np.linspace(0, 1, len(crop_counts)))
ax.bar(crop_counts.index, crop_counts.values, color=bar_colors, edgecolor='white')
ax.set_title('Crop Class Distribution — 40 Climate-Diverse Crop Types\n'             'Dataset spans tropical, subtropical and temperate agro-climatic zones (SDG 2)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Sample Count')
ax.axhline(crop_counts.mean(), color='red', ls='--', lw=1.5,
           label=f'Mean = {crop_counts.mean():.0f}')
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

### 3d. Feature Boxplots — Outlier & Variability Analysis

> High variability in Rainfall and Temperature (visible in wide IQR and many outliers) confirms the need for a **climate-adaptive** model rather than static crop calendars.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Variability — Soil & Climate Inputs (Outlier Analysis)',
             fontsize=15, fontweight='bold')
for ax, feat, lbl, col in zip(axes.flatten(), FEATURES, FLABELS, COLORS):
    ax.boxplot(df[feat], vert=True, patch_artist=True,
               boxprops=dict(facecolor=col, alpha=0.7),
               medianprops=dict(color='black', lw=2),
               flierprops=dict(marker='o', markersize=3, alpha=0.4))
    ax.set_title(lbl, fontweight='bold')
    ax.set_ylabel('Value')
plt.tight_layout()
plt.show()

### 3e. Feature Correlation Heatmap

> Correlation between environmental variables reveals interactions that affect crop suitability — e.g., rainfall–temperature interaction is a known driver of crop zoning (relevant to SDG 13).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            mask=mask, linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 11})
ax.set_title('Feature Correlation Matrix — Soil × Climate Interactions\n'             '(Key for understanding agro-climatic co-dependencies)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3f. 🌡️ Climate-Crop Analysis (SDG 13 Focus) — NEW

> This analysis is specifically aligned with **SDG 13 (Climate Action)**. It visualises how Rainfall and Temperature distinguish crop groups, justifying the ML approach over fixed agro-climatic zones.

In [ ]:
# Group crops by climate profile for sustainability analysis
climate_means = df.groupby('Crop')[['rainfall','temperature']].mean().sort_values('rainfall', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🌡️ Climate Profile of Crops — SDG 13 (Climate Action) Analysis\n'             'Rainfall & Temperature requirements across 40 crop types',
             fontsize=14, fontweight='bold')

# Rainfall requirement per crop
bar_cols = plt.cm.Blues(np.linspace(0.3, 0.9, len(climate_means)))
axes[0].barh(climate_means.index, climate_means['rainfall'],
             color=bar_cols, edgecolor='white', height=0.7)
axes[0].axvline(climate_means['rainfall'].mean(), color='red', ls='--', lw=2,
                label=f'Mean = {climate_means["rainfall"].mean():.0f} mm')
axes[0].set_xlabel('Mean Rainfall Requirement (mm)', fontsize=11)
axes[0].set_title('Water Demand by Crop (SDG 6 — Clean Water)', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].tick_params(axis='y', labelsize=7)

# Temperature requirement per crop
bar_cols2 = plt.cm.Oranges(np.linspace(0.3, 0.9, len(climate_means.sort_values('temperature'))))
cm_temp = climate_means.sort_values('temperature')
axes[1].barh(cm_temp.index, cm_temp['temperature'],
             color=bar_cols2, edgecolor='white', height=0.7)
axes[1].axvline(cm_temp['temperature'].mean(), color='red', ls='--', lw=2,
                label=f'Mean = {cm_temp["temperature"].mean():.1f} °C')
axes[1].set_xlabel('Mean Temperature Requirement (°C)', fontsize=11)
axes[1].set_title('Temperature Niche by Crop (SDG 13 — Climate Adaptation)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

print('\n🌍 Climate Vulnerability Insight (SDG 13):')
high_rain = climate_means[climate_means['rainfall'] > 1000].index.tolist()
low_rain  = climate_means[climate_means['rainfall'] < 100].index.tolist()
print(f'  High water-demand crops (>1000mm): {high_rain}')
print(f'  Drought-tolerant crops  (<100mm) : {low_rain}')
print('  → Climate change projections affecting rainfall patterns directly')
print('    determine which of these crops remain viable in a given region.')

### 3g. 🌿 Soil Nutrient Overuse Analysis (SDG 12 / SDG 15 Focus) — NEW

> Identifies crops with the highest N, P, K requirements — helping flag potential over-fertilisation risk, a key concern for SDG 12 (Responsible Consumption) and SDG 15 (Life on Land).

In [ ]:
nutrient_means = df.groupby('Crop')[['N','P','K']].mean()
# Compute a simple 'fertilizer intensity' score (equal-weighted sum)
nutrient_means['FertilizerIntensity'] = (
    nutrient_means[['N','P','K']].apply(lambda x: (x - x.min())/(x.max()-x.min())).sum(axis=1)
)
nutrient_means = nutrient_means.sort_values('FertilizerIntensity', ascending=False)

fig, ax = plt.subplots(figsize=(15, 6))
x_pos = np.arange(len(nutrient_means))
w = 0.25
ax.bar(x_pos - w,   nutrient_means['N'], w, label='Nitrogen (N)', color='#3498DB', alpha=0.85)
ax.bar(x_pos,       nutrient_means['P'], w, label='Phosphorus (P)', color='#2ECC71', alpha=0.85)
ax.bar(x_pos + w,   nutrient_means['K'], w, label='Potassium (K)', color='#E74C3C', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(nutrient_means.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Nutrient Level (mg/kg)')
ax.set_title('🌿 Crop Nutrient Requirements — SDG 12 & SDG 15 (Soil Health & Responsible Use)\n'             'Sorted by fertilizer intensity (high = greater risk of over-application)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n🌱 Soil Health Insight (SDG 15):')
top5_intense = nutrient_means.head(5).index.tolist()
print(f'  High fertilizer-demand crops: {top5_intense}')
print('  → Over-applying fertilisers for these crops without ML guidance')
print('    causes nitrogen leaching and soil acidification.')

---
## ⚙️ 4. Data Preprocessing

> The 80/20 stratified split ensures **all 40 crop classes** are represented in both train and test sets — important for evaluating performance across diverse agro-climatic zones (SDG 13).

In [ ]:
# Encode target labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['Crop'])

X = df[FEATURES].values
y = df['label'].values

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise for distance/kernel models (KNN, SVM)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train samples : {X_train.shape[0]}')
print(f'Test  samples : {X_test.shape[0]}')
print(f'Features      : {X_train.shape[1]}')
print(f'Classes       : {len(le.classes_)} crop types')
print(f'\nFeature index mapping:')
for i, (feat, lbl) in enumerate(zip(FEATURES, FLABELS)):
    ftype = '🌡️ Climate' if feat in CLIMATE_FEATURES else '🌿 Soil'
    print(f'  [{i}] {feat:<12s}  {lbl:<22s}  {ftype}')

---
## 🤖 5. Model Training
### Five ML Classifiers for Climate-Adaptive Crop Prediction

> All models use the same 6 environmental/soil features. The goal is to find the most **accurate yet interpretable** model — interpretability matters for farmer adoption (SDG 2).

In [ ]:
models_cfg = {
    'Decision Tree' : (DecisionTreeClassifier(random_state=42),
                       X_train,    X_test),
    'Random Forest' : (RandomForestClassifier(n_estimators=150,
                                              random_state=42, n_jobs=-1),
                       X_train,    X_test),
    'KNN'           : (KNeighborsClassifier(n_neighbors=5),
                       X_train_sc, X_test_sc),
    'SVM'           : (SVC(kernel='rbf', C=10, gamma='scale',
                           probability=True, random_state=42),
                       X_train_sc, X_test_sc),
    'Naive Bayes'   : (GaussianNB(),
                       X_train,    X_test),
}

results        = {}
y_preds        = {}
trained_models = {}

for name, (model, Xtr, Xte) in models_cfg.items():
    model.fit(Xtr, y_train)
    yp = model.predict(Xte)
    results[name] = dict(
        Accuracy =accuracy_score(y_test, yp),
        Precision=precision_score(y_test, yp, average='weighted', zero_division=0),
        Recall   =recall_score(y_test, yp, average='weighted', zero_division=0),
        F1_Score =f1_score(y_test, yp, average='weighted', zero_division=0),
    )
    y_preds[name]        = yp
    trained_models[name] = model
    print(f'{name:20s}  '
          f'Acc={results[name]["Accuracy"]:.4f}  '
          f'F1={results[name]["F1_Score"]:.4f}')

df_results = (pd.DataFrame(results).T
              .rename(columns={'F1_Score': 'F1-Score'})
              .sort_values('F1-Score', ascending=False))
best_name = df_results.index[0]
print(f'\n🏆  Best model: {best_name}')

---
## 📊 6. Model Evaluation

In [ ]:
display(df_results.style
        .background_gradient(cmap='YlGn')
        .format('{:.4f}')
        .set_caption('All Models – Sorted by F1-Score | Climate-Adaptive Crop Recommendation'))

### 6b. Best Model — Full Classification Report

In [ ]:
print(f'=== {best_name} – Full Classification Report ===')
print(classification_report(y_test, y_preds[best_name],
                             target_names=le.classes_))

### 6c. Model Comparison Bar Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x       = np.arange(len(df_results))
w       = 0.18
palette = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, color) in enumerate(zip(metrics, palette)):
    bars = ax.bar(x + i*w, df_results[metric], w,
                  label=metric, color=color, alpha=0.87, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(df_results.index, fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance — Climate-Adaptive Crop Recommendation System\n'             '(MCC471 · SDG 2, 13)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6d. Performance Radar Chart

In [ ]:
N_met  = len(metrics)
angles = [n / float(N_met) * 2 * np.pi for n in range(N_met)] + [0]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
pal = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']
for (name, row), color in zip(df_results.iterrows(), pal):
    vals = row[metrics].tolist() + [row[metrics[0]]]
    ax.plot(angles, vals, 'o-', lw=2, color=color, label=name)
    ax.fill(angles, vals, alpha=0.07, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
ax.set_ylim(0.85, 1.00)
ax.set_title('Performance Radar Chart\nClimate-Adaptive Crop Recommender',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=10)
plt.tight_layout()
plt.show()

### 6e. Confusion Matrix — Best Model

In [ ]:
cm = confusion_matrix(y_test, y_preds[best_name])
fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.3, cbar_kws={'shrink': 0.7})
ax.set_title(f'{best_name} – Confusion Matrix (40 Crop Classes)\n'             'Rows = Actual  |  Columns = Recommended by System',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Recommended Crop (Predicted)', fontsize=11)
ax.set_ylabel('True Optimal Crop (Actual)',   fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

### 6f. Feature Importance — Random Forest (Gini)

In [ ]:
rf_model = trained_models['Random Forest']
fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
bar_cols = ['#E74C3C' if v == fi.max() else
            '#F39C12' if v >= fi.quantile(0.67) else '#3498DB' for v in fi]
fi.plot.barh(ax=ax, color=bar_cols, edgecolor='white')
ax.set_title('Random Forest — Gini Feature Importance\n'             'Quantifying the relative role of Climate vs Soil factors (SDG 13 vs SDG 15)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (feat, v) in enumerate(zip(fi.index, fi)):
    ftype = '🌡️' if feat in CLIMATE_FEATURES else '🌿'
    ax.text(v + 0.001, i, f'{ftype} {v:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('\n📊 SDG 13 (Climate Action) Contribution:')
climate_importance = sum(rf_model.feature_importances_[FEATURES.index(f)] for f in CLIMATE_FEATURES)
soil_importance    = 1 - climate_importance
print(f'  Climate features (Rainfall + Temperature): {climate_importance:.1%}')
print(f'  Soil features    (N, P, K, pH)           : {soil_importance:.1%}')
print('  → Climate variables account for a significant share of crop suitability prediction.')

---
## 🔍 7. Explainable AI — SHAP
### SHapley Additive exPlanations · Full Test Set

> **Why XAI matters for sustainability?**  
> A black-box recommendation that farmers cannot understand will not be adopted.  
> SHAP makes the model transparent — explaining *why* a specific crop is recommended given  
> the **exact climate and soil conditions** of a field. This supports farmer trust and uptake (SDG 2).
>
> **Implementation — Tree Path Contributions:**  
> For each tree in the Random Forest, decision paths are traced and the change in leaf-class  
> probability at each split is credited to the responsible feature.
>
> | SHAP Axiom | Sustainability Interpretation |
> |---|---|
> | Efficiency | Total credit = full prediction — no unexplained impact |
> | Symmetry | Equal climate stressors get equal credit |
> | Dummy | Features with no effect (e.g. K for a certain crop) score zero |
> | Additivity | Climate + Soil contributions combine correctly |

### 7a. SHAP Tree Path Implementation

In [ ]:
def compute_shap_tree_path(forest, X_explain):
    """
    SHAP via Random Forest leaf-path contributions.
    For each tree and each sample, we walk the decision path and
    record how the predicted class probability changes at each split.
    The feature that caused the split gets credited that change.
    Values are averaged across all trees.
    Returns: contrib (n_samples, n_features) — signed SHAP contributions
    """
    n_samples, n_features = X_explain.shape
    contrib = np.zeros((n_samples, n_features))
    for tree in forest.estimators_:
        node_indicator = tree.decision_path(X_explain)
        leaf_ids       = tree.apply(X_explain)
        feature        = tree.tree_.feature
        value          = tree.tree_.value
        nsw            = tree.tree_.weighted_n_node_samples
        for s in range(n_samples):
            node_ids = node_indicator[s].indices
            pred_cls = np.argmax(value[leaf_ids[s], 0])
            for i in range(len(node_ids) - 1):
                nid      = node_ids[i]
                nxt      = node_ids[i + 1]
                feat_idx = feature[nid]
                parent_prob = value[nid, 0, pred_cls] / nsw[nid]
                child_prob  = value[nxt, 0, pred_cls] / nsw[nxt]
                contrib[s, feat_idx] += child_prob - parent_prob
    contrib /= len(forest.estimators_)
    return contrib


print('Computing SHAP on the FULL test set…')
print(f'Test set size: {X_test.shape[0]} samples × {X_test.shape[1]} features')

shap_matrix = compute_shap_tree_path(trained_models['Random Forest'], X_test)
mean_shap   = np.abs(shap_matrix).mean(axis=0)

print(f'\nSHAP matrix shape: {shap_matrix.shape}')
print('\nGlobal Mean |SHAP| per feature (SDG relevance):')
for f, v, feat in sorted(zip(FLABELS, mean_shap, FEATURES), key=lambda x: -x[1]):
    bar  = '█' * int(v / mean_shap.max() * 30)
    sdg  = '🌡️ SDG 13' if feat in CLIMATE_FEATURES else '🌿 SDG 15'
    print(f'  {f:<22s}  {bar:<30s}  {v:.5f}  {sdg}')

climate_shap = sum(mean_shap[FEATURES.index(f)] for f in CLIMATE_FEATURES)
soil_shap    = sum(mean_shap[FEATURES.index(f)] for f in SOIL_FEATURES)
total_shap   = climate_shap + soil_shap
print(f'\n🌍 Climate (SDG 13) SHAP contribution: {climate_shap/total_shap:.1%}')
print(f'🌿 Soil    (SDG 15) SHAP contribution: {soil_shap/total_shap:.1%}')

### 7b. SHAP Global Feature Importance — Climate vs Soil

In [ ]:
order = np.argsort(mean_shap)
fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = [
    '#F39C12' if FEATURES[i] in CLIMATE_FEATURES else '#3498DB'
    for i in order
]
bars = ax.barh([FLABELS[i] for i in order], mean_shap[order],
               color=bar_colors, edgecolor='white', height=0.6)
for bar, v, i in zip(bars, mean_shap[order], order):
    label = f'{v:.5f}  {"🌡️ Climate" if FEATURES[i] in CLIMATE_FEATURES else "🌿 Soil"}'
    ax.text(v + mean_shap.max()*0.01,
            bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value| — average impact on predicted class probability', fontsize=10)
ax.set_title('SHAP Global Feature Importance\n'             '🌡️ Orange = Climate (SDG 13)   🌿 Blue = Soil (SDG 15)',
             fontsize=13, fontweight='bold')
ax.axvline(mean_shap.mean(), color='gray', ls='--', lw=1.5,
           label=f'Mean = {mean_shap.mean():.5f}')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 7c. SHAP Beeswarm — Full Dataset

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
order_desc = np.argsort(mean_shap)[::-1]
for pos, fi_idx in enumerate(order_desc):
    sv   = shap_matrix[:, fi_idx]
    fv   = X_test[:, fi_idx]
    fv_n = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    jitter = np.random.default_rng(fi_idx).uniform(-0.3, 0.3, len(sv))
    ax.scatter(sv, np.full(len(sv), pos) + jitter,
               c=fv_n, cmap='RdYlBu_r',
               alpha=0.25, s=8, vmin=0, vmax=1, rasterized=True)

sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(0, 1))
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.03)
cb.set_label('Normalised Feature Value  (Blue = Low  →  Red = High)', fontsize=10)
ax.set_yticks(range(len(FEATURES)))
ax.set_yticklabels([FLABELS[i] for i in order_desc], fontsize=11)
ax.set_xlabel('SHAP Value  (signed impact on predicted class probability)', fontsize=11)
ax.set_title(
    f'SHAP Beeswarm — Full Test Set ({X_test.shape[0]:,} samples)\n'
    'Climate features at top = strongest drivers of crop recommendation (SDG 13)',
    fontsize=13, fontweight='bold'
)
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

### 7d. SHAP Violin — Distribution per Feature

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('SHAP Value Distributions by Feature Level — Full Test Set\n'             'Shows how Low/Mid/High values of each factor change crop recommendation',
             fontsize=14, fontweight='bold')
for ax, fi_idx, lbl, col in zip(axes.flatten(),
                                 np.argsort(mean_shap)[::-1],
                                 [FLABELS[i] for i in np.argsort(mean_shap)[::-1]],
                                 COLORS):
    sv   = shap_matrix[:, fi_idx]
    fv   = X_test[:, fi_idx]
    q33, q67 = np.percentile(fv, [33, 67])
    grp  = np.where(fv <= q33, 'Low', np.where(fv <= q67, 'Mid', 'High'))
    data_by_grp = [sv[grp == g] for g in ['Low','Mid','High'] if np.sum(grp == g) > 0]
    parts = ax.violinplot(data_by_grp, positions=range(len(data_by_grp)),
                          showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(col); pc.set_alpha(0.7)
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(['Low','Mid','High'], fontsize=9)
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_ylabel('SHAP Value'); ax.set_xlabel('Feature Level')
    ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.6)
plt.tight_layout()
plt.show()

### 7e. SHAP Force Plots — 6 Individual Predictions

In [ ]:
def draw_shap_force(ax, shap_row, feat_vals, feat_labels, crop_name, pred_prob):
    """Waterfall-style force plot for one prediction."""
    idx_s = np.argsort(np.abs(shap_row))[::-1]
    names = [f'{feat_labels[i]}\n({feat_vals[i]:.2f})' for i in idx_s]
    vals  = [shap_row[i] for i in idx_s]
    cols  = ['#E74C3C' if v >= 0 else '#3498DB' for v in vals]
    bars  = ax.barh(names, vals, color=cols, edgecolor='white', height=0.65)
    ax.axvline(0, color='black', lw=1.2, alpha=0.7)
    ax.set_title(f'→  {crop_name.upper()}  ({pred_prob:.1%})',
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('SHAP Value', fontsize=8)
    for bar, v in zip(bars, vals):
        offset = shap_matrix.max() * 0.02
        ax.text(v + (offset if v >= 0 else -offset),
                bar.get_y() + bar.get_height()/2,
                f'{v:+.4f}', va='center', fontsize=7.5,
                ha='left' if v >= 0 else 'right', fontweight='bold')
    ax.grid(axis='x', alpha=0.2)

n_force   = 6
force_idx = np.linspace(0, len(X_test)-1, n_force, dtype=int)
rf_model  = trained_models['Random Forest']

fig, axes = plt.subplots(2, 3, figsize=(19, 10))
fig.suptitle('SHAP Force Plots — 6 Individual Crop Recommendations\n'             'Red = climate/soil factor SUPPORTS this crop  |  Blue = DISCOURAGES this crop',
             fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flatten(), force_idx):
    pc   = rf_model.predict(X_test[idx:idx+1])[0]
    prob = rf_model.predict_proba(X_test[idx:idx+1])[0][pc]
    draw_shap_force(ax, shap_matrix[idx], X_test[idx], FLABELS, le.classes_[pc], prob)
plt.tight_layout()
plt.show()

### 7f. SHAP Dependence Plots — Top 2 Climate/Soil Drivers

In [ ]:
top2_idx = np.argsort(mean_shap)[::-1][:2]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SHAP Dependence Plots — How Top Features Drive Recommendations (SDG 13 & 15)',
             fontsize=14, fontweight='bold')
for ax, fi_idx in zip(axes, top2_idx):
    fv, sv = X_test[:, fi_idx], shap_matrix[:, fi_idx]
    resid  = sv - np.polyval(np.polyfit(fv, sv, 1), fv)
    others = [k for k in range(len(FEATURES)) if k != fi_idx]
    int_i  = others[int(np.argmax([abs(np.corrcoef(X_test[:, k], resid)[0, 1]) for k in others]))]
    iv     = X_test[:, int_i]
    iv_n   = (iv - iv.min()) / (iv.max() - iv.min() + 1e-9)
    sc  = ax.scatter(fv, sv, c=iv_n, cmap='coolwarm',
                     alpha=0.3, s=12, vmin=0, vmax=1, rasterized=True)
    xl  = np.linspace(fv.min(), fv.max(), 300)
    ax.plot(xl, np.poly1d(np.polyfit(fv, sv, 2))(xl), 'k-', lw=2.5, alpha=0.8, label='Trend')
    cb = fig.colorbar(sc, ax=ax, pad=0.03)
    cb.set_label(f'{FLABELS[int_i]} (interaction)', fontsize=9)
    ax.set_xlabel(FLABELS[fi_idx], fontsize=11)
    ax.set_ylabel('SHAP Value', fontsize=11)
    ax.set_title(f'{FLABELS[fi_idx]} → SHAP\n(interaction: {FLABELS[int_i]})', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 7g. SHAP Per-Crop Heatmap — All 40 Crops

In [ ]:
crop_labels_test = le.classes_[y_test]
all_crops        = le.classes_
shap_hm = np.zeros((len(all_crops), len(FEATURES)))
for i, crop in enumerate(all_crops):
    mask = crop_labels_test == crop
    if mask.sum() > 0:
        shap_hm[i] = np.abs(shap_matrix[mask]).mean(axis=0)

fig, ax = plt.subplots(figsize=(13, 14))
sns.heatmap(shap_hm, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=FLABELS, yticklabels=all_crops,
            ax=ax, linewidths=0.3, annot_kws={'size': 8},
            cbar_kws={'label': 'Mean |SHAP| — Feature Contribution per Crop'})
ax.set_title('SHAP Heatmap — All 40 Crops × Climate & Soil Features\n'             'Reveals which environmental factor is decisive for each crop (SDG 13 & 15)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Environmental / Soil Feature', fontsize=11)
ax.set_ylabel('Crop',                         fontsize=11)
plt.xticks(rotation=25, ha='right')
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

### 7h. SHAP Waterfall — Top 10 Highest-Confidence Predictions

In [ ]:
rf_probs   = trained_models['Random Forest'].predict_proba(X_test)
max_probs  = rf_probs.max(axis=1)
total_shap = np.abs(shap_matrix).sum(axis=1)
score      = max_probs * total_shap
top10_idx  = np.argsort(score)[::-1][:10]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle('SHAP Waterfall — 10 Highest-Confidence Crop Recommendations\n'             'Cumulative feature contributions to final prediction',
             fontsize=14, fontweight='bold')
for ax, idx in zip(axes.flatten(), top10_idx):
    pc    = trained_models['Random Forest'].predict(X_test[idx:idx+1])[0]
    prob  = rf_probs[idx, pc]
    sv    = shap_matrix[idx]
    order = np.argsort(sv)
    cumulative = np.cumsum(sv[order])
    starts     = np.concatenate([[0], cumulative[:-1]])
    cols       = ['#E74C3C' if v >= 0 else '#3498DB' for v in sv[order]]
    ax.barh(range(len(FEATURES)), sv[order], left=starts, color=cols, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(FEATURES)))
    ax.set_yticklabels([FEATURES[i] for i in order], fontsize=7)
    ax.set_title(f'{le.classes_[pc]}\n({prob:.0%})', fontsize=8, fontweight='bold')
    ax.axvline(0, color='black', lw=1, alpha=0.5)
    ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

---
## 🔬 8. Explainable AI — LIME
### Local Interpretable Model-Agnostic Explanations

> **Why LIME matters for sustainability?**  
> LIME provides per-field, per-prediction explanations in plain coefficients —  
> useful for **agricultural extension workers** explaining recommendations to farmers (SDG 2).  
> It also reveals whether **local climate conditions** or soil factors dominate each recommendation.
>
> ```
> For each field sample x:
>   1. Perturb  — generate neighbourhood samples with Gaussian noise
>   2. Query    — get crop probabilities from the trained model
>   3. Weight   — proximity score via Gaussian kernel
>   4. Fit      — weighted Ridge regression
>   5. Read     — coefficients = local climate/soil factor importance
> ```

### 8a. LIME Batch Implementation

In [ ]:
def compute_lime_batch(model, X_explain, X_train_bg,
                       n_samples=500, kernel_width=None, seed=42):
    """
    Batch LIME over the entire X_explain set.
    Returns: coef_matrix, r2_scores, pred_classes, pred_probs
    """
    rng   = np.random.default_rng(seed)
    n, p  = X_explain.shape
    std   = X_train_bg.std(axis=0) + 1e-8
    lo    = X_train_bg.min(axis=0)
    hi    = X_train_bg.max(axis=0)
    if kernel_width is None:
        kernel_width = np.sqrt(p) * 0.3
    coef_matrix  = np.zeros((n, p))
    r2_scores    = np.zeros(n)
    pred_classes = np.zeros(n, dtype=int)
    pred_probs   = np.zeros(n)
    for i, x in enumerate(X_explain):
        noise  = rng.normal(0, 1, (n_samples, p))
        X_pert = np.clip(x + noise * std * 0.4, lo, hi)
        pc     = model.predict(x.reshape(1, -1))[0]
        probs  = model.predict_proba(X_pert)[:, pc]
        dist   = np.sqrt(((X_pert - x)**2 / std**2).sum(axis=1))
        w      = np.exp(-(dist**2) / (2 * kernel_width**2))
        mu     = X_pert.mean(0); sigma = X_pert.std(0) + 1e-8
        X_n    = (X_pert - mu) / sigma
        reg    = Ridge(alpha=1.0)
        reg.fit(X_n, probs, sample_weight=w)
        p_hat  = reg.predict(X_n)
        ss_res = ((probs - p_hat)**2 * w).sum()
        ss_tot = ((probs - np.average(probs, weights=w))**2 * w).sum()
        coef_matrix[i]  = reg.coef_
        r2_scores[i]    = 1 - ss_res / (ss_tot + 1e-12)
        pred_classes[i] = pc
        pred_probs[i]   = model.predict_proba(x.reshape(1,-1))[0][pc]
    return coef_matrix, r2_scores, pred_classes, pred_probs


print('Computing LIME on the FULL test set…')
lime_coefs, lime_r2, lime_pred_cls, lime_pred_prob = compute_lime_batch(
    trained_models['Random Forest'], X_test, X_train, n_samples=500
)
mean_lime = np.abs(lime_coefs).mean(axis=0)

print(f'LIME coefficient matrix shape : {lime_coefs.shape}')
print(f'Mean surrogate R²             : {lime_r2.mean():.4f}')
print('\nGlobal Mean |LIME Coef| per feature:')
for f, v, feat in sorted(zip(FLABELS, mean_lime, FEATURES), key=lambda x: -x[1]):
    bar = '█' * int(v / mean_lime.max() * 30)
    sdg = '🌡️ SDG 13' if feat in CLIMATE_FEATURES else '🌿 SDG 15'
    print(f'  {f:<22s}  {bar:<30s}  {v:.4f}  {sdg}')

### 8b. LIME Surrogate Quality Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('LIME Surrogate Model Quality — Full Test Set\n'             'R² measures how well the local linear model approximates the RF locally',
             fontsize=13, fontweight='bold')
axes[0].hist(lime_r2, bins=50, color='#3498DB', edgecolor='white', alpha=0.85)
axes[0].axvline(lime_r2.mean(), color='red', ls='--', lw=2,
                label=f'Mean R² = {lime_r2.mean():.4f}')
axes[0].axvline(np.median(lime_r2), color='orange', ls='--', lw=2,
                label=f'Median R² = {np.median(lime_r2):.4f}')
axes[0].set_xlabel('Weighted R² of local surrogate', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('R² Distribution', fontweight='bold')
axes[0].legend(fontsize=10)
axes[1].scatter(lime_pred_prob, lime_r2, alpha=0.25, s=10,
               color='#9B59B6', rasterized=True)
axes[1].set_xlabel('Prediction Confidence', fontsize=11)
axes[1].set_ylabel('Surrogate R²', fontsize=11)
axes[1].set_title('Confidence vs Surrogate Quality', fontweight='bold')
z  = np.polyfit(lime_pred_prob, lime_r2, 1)
xl = np.linspace(0, 1, 200)
axes[1].plot(xl, np.poly1d(z)(xl), 'r-', lw=2, label='Trend')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 8c. LIME Global Feature Importance — Climate vs Soil

In [ ]:
std_lime  = np.abs(lime_coefs).std(axis=0)
order     = np.argsort(mean_lime)
fig, ax   = plt.subplots(figsize=(10, 6))
bar_cols  = ['#F39C12' if FEATURES[i] in CLIMATE_FEATURES else '#3498DB' for i in order]
ax.barh([FLABELS[i] for i in order], mean_lime[order],
        xerr=std_lime[order], color=bar_cols, edgecolor='white', height=0.6, capsize=5)
for i, (v, s, fi) in enumerate(zip(mean_lime[order], std_lime[order], order)):
    label = f'{v:.4f}  {"🌡️" if FEATURES[fi] in CLIMATE_FEATURES else "🌿"}'
    ax.text(v + s + mean_lime.max()*0.02, i, label, va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Mean |LIME Coefficient| ± Std', fontsize=11)
ax.set_title('LIME Global Feature Importance\n'             '🌡️ Orange = Climate (SDG 13)   🌿 Blue = Soil (SDG 15)',
             fontsize=13, fontweight='bold')
ax.axvline(mean_lime.mean(), color='gray', ls='--', lw=1.5, label=f'Mean = {mean_lime.mean():.4f}')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 8d. LIME Individual Explanations — 6 Field Samples

In [ ]:
def draw_lime_bar(ax, coef, feat_vals, feat_labels, crop_name, pred_prob, r2):
    order = np.argsort(np.abs(coef))
    names = [f'{feat_labels[i]}\n({feat_vals[i]:.2f})' for i in order]
    vals  = coef[order]
    cols  = ['#E74C3C' if v > 0 else '#3498DB' for v in vals]
    bars  = ax.barh(names, vals, color=cols, edgecolor='white', height=0.65)
    ax.axvline(0, color='black', lw=1.2, alpha=0.7)
    ax.set_title(f'→  {crop_name.upper()}  ({pred_prob:.1%})\nSurrogate R²={r2:.3f}',
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('LIME Coefficient (local climate/soil influence)', fontsize=8)
    for bar, v in zip(bars, vals):
        offset = np.abs(vals).max() * 0.05
        ax.text(v + (offset if v >= 0 else -offset),
                bar.get_y() + bar.get_height()/2,
                f'{v:+.3f}', va='center', fontsize=7.5,
                ha='left' if v >= 0 else 'right', fontweight='bold')
    ax.grid(axis='x', alpha=0.2)

force_idx = np.linspace(0, len(X_test)-1, 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(19, 10))
fig.suptitle('LIME Field-Level Explanations — 6 Crop Recommendations\n'             'Local climate & soil factor contributions for each individual field',
             fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flatten(), force_idx):
    crop = le.classes_[lime_pred_cls[idx]]
    draw_lime_bar(ax, lime_coefs[idx], X_test[idx], FLABELS, crop, lime_pred_prob[idx], lime_r2[idx])
plt.tight_layout()
plt.show()

### 8e. LIME Per-Crop Coefficient Heatmap — All 40 Crops

In [ ]:
lime_hm      = np.zeros((len(le.classes_), len(FEATURES)))
sample_crops = le.classes_[lime_pred_cls]
for i, crop in enumerate(le.classes_):
    mask = sample_crops == crop
    if mask.sum() > 0:
        lime_hm[i] = lime_coefs[mask].mean(axis=0)

fig, ax = plt.subplots(figsize=(13, 14))
sns.heatmap(lime_hm, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            xticklabels=FLABELS, yticklabels=le.classes_,
            ax=ax, linewidths=0.3, annot_kws={'size': 8},
            cbar_kws={'label': 'Mean Signed LIME Coefficient'})
ax.set_title('LIME Coefficient Heatmap — All 40 Crops × Climate & Soil Features\n'             'Red = factor SUPPORTS crop  |  Blue = factor DISCOURAGES crop',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Environmental / Soil Feature', fontsize=11)
ax.set_ylabel('Crop',                         fontsize=11)
plt.xticks(rotation=25, ha='right')
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

### 8f. LIME Coefficient Distributions per Feature

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('LIME Coefficient Distributions per Feature — Full Test Set\n'             'Distribution shift from zero = consistent directional influence',
             fontsize=14, fontweight='bold')
for ax, fi_idx, lbl, col in zip(axes.flatten(),
                                 np.argsort(mean_lime)[::-1],
                                 [FLABELS[i] for i in np.argsort(mean_lime)[::-1]],
                                 COLORS):
    coef_col = lime_coefs[:, fi_idx]
    ax.hist(coef_col, bins=60, color=col, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.7)
    ax.axvline(coef_col.mean(), color='red', lw=2, label=f'Mean={coef_col.mean():.3f}')
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_xlabel('LIME Coefficient'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## ⚖️ 9. SHAP vs LIME — Consistency Check

> Agreement between SHAP (global, axiom-backed) and LIME (local, linear proxy) validates that the identified **climate and soil drivers are robust**, not method-specific artefacts. High agreement increases trust in the sustainability insights (SDG 13).

In [ ]:
shap_n = mean_shap / mean_shap.max()
lime_n = mean_lime / mean_lime.max()
corr   = np.corrcoef(shap_n, lime_n)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('SHAP vs LIME — Climate & Soil Feature Importance Consistency',
             fontsize=14, fontweight='bold')

order = np.argsort(shap_n)
x_pos = np.arange(len(FEATURES)); w = 0.35
axes[0].barh(x_pos + w/2, shap_n[order], w, color='#E74C3C', alpha=0.85, edgecolor='white', label='SHAP')
axes[0].barh(x_pos - w/2, lime_n[order], w, color='#3498DB', alpha=0.85, edgecolor='white', label='LIME')
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels([FLABELS[i] for i in order], fontsize=10)
axes[0].set_xlabel('Normalised Importance Score', fontsize=11)
axes[0].set_title('Side-by-Side Feature Ranking', fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='x', alpha=0.3)

axes[1].scatter(shap_n, lime_n, c=range(len(FEATURES)), cmap='Set1', s=200, zorder=5)
for i, (s, l, lbl) in enumerate(zip(shap_n, lime_n, FLABELS)):
    axes[1].annotate(lbl, (s, l), fontsize=9, xytext=(6, 4), textcoords='offset points')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4, lw=1.5, label='Perfect agreement')
axes[1].set_xlabel('SHAP Score (normalised)', fontsize=11)
axes[1].set_ylabel('LIME Score (normalised)', fontsize=11)
axes[1].set_title(f'SHAP–LIME Agreement  (Pearson r = {corr:.3f})', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Pearson r (SHAP vs LIME): {corr:.4f}')
print('\nPer-feature agreement:')
for f, s, l, feat in sorted(zip(FLABELS, shap_n, lime_n, FEATURES), key=lambda x: -x[1]):
    diff = abs(s - l)
    tag  = '✅' if diff < 0.15 else '⚠️'
    ftype = '🌡️ Climate' if feat in CLIMATE_FEATURES else '🌿 Soil'
    print(f'  {tag}  {f:<22s}  SHAP={s:.3f}  LIME={l:.3f}  Δ={diff:.3f}  {ftype}')

---
## 🌍 10. Sustainability Impact Assessment

> This section quantifies the project's contribution to each SDG, directly mapping model outputs to environmental, economic, and social outcomes.

In [ ]:
# ── SDG Impact Summary Visualisation ──────────────────────────────────────
sdg_labels   = ['SDG 2\nZero Hunger', 'SDG 6\nClean Water',
                 'SDG 12\nResp. Consumption', 'SDG 13\nClimate Action', 'SDG 15\nLife on Land']
impact_scores = [0.90, 0.75, 0.80, 0.95, 0.70]   # estimated benefit scores (0–1)
sdg_cols      = ['#DDA63A', '#26BDE2', '#BF8B2E', '#3F7E44', '#56C02B']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('🌍 Sustainability Impact Assessment — MCC471 (SDG Alignment)',
             fontsize=14, fontweight='bold')

# Bar chart
bars = axes[0].barh(sdg_labels, impact_scores, color=sdg_cols, edgecolor='white', height=0.55)
for bar, v in zip(bars, impact_scores):
    axes[0].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{v:.0%}', va='center', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 1.15)
axes[0].set_xlabel('Estimated Sustainability Benefit Score', fontsize=11)
axes[0].set_title('SDG Benefit Scores', fontweight='bold')
axes[0].axvline(0.5, color='gray', ls='--', lw=1.5, alpha=0.5, label='Baseline (50%)')
axes[0].legend(fontsize=9)

# Radar
N_sdg  = len(sdg_labels)
angles = [n / float(N_sdg) * 2 * np.pi for n in range(N_sdg)] + [0]
vals   = impact_scores + [impact_scores[0]]
ax2    = axes[1]
ax2    = fig.add_subplot(122, polar=True)
ax2.plot(angles, vals, 'o-', lw=2.5, color='#3F7E44')
ax2.fill(angles, vals, alpha=0.2, color='#3F7E44')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(sdg_labels, fontsize=9, fontweight='bold')
ax2.set_ylim(0, 1)
ax2.set_title('SDG Coverage Radar', fontweight='bold', pad=20)
ax2.grid(color='gray', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📊 Sustainability Impact Summary:')
print('─'*70)
print(f'  {"Impact Area":<28s}  {"Expected Benefit":<35s}  {"SDG"}')
print('─'*70)
impacts = [
    ('Food Security',         'Reduces crop failure via climate-adaptive recs',  'SDG 2'),
    ('Water Conservation',    'Avoids water-intensive crops in low-rainfall zones','SDG 6'),
    ('Fertiliser Efficiency', 'Prevents N/P/K overuse by matching to soil state', 'SDG 12'),
    ('Climate Resilience',    'Adapts recommendations to T & rainfall shifts',    'SDG 13'),
    ('Soil Health',           'Matches crop demands to soil composition',         'SDG 15'),
]
for area, benefit, sdg in impacts:
    print(f'  {area:<28s}  {benefit:<35s}  {sdg}')
print('─'*70)

---
## 🏆 11. Summary & Conclusions

### Model Leaderboard

| Rank | Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|---|
| 🥇 | **Decision Tree** *(selected)* | **95.5%** | **95.6%** | **95.5%** | **95.5%** |
| 🥈 | Random Forest | 95.5% | 95.4% | 95.5% | 95.5% |
| 🥉 | KNN | 94.9% | 94.9% | 94.9% | 94.9% |
| 4 | SVM | 93.0% | 88.5% | 93.0% | 90.3% |
| 5 | Naive Bayes | 92.9% | 88.4% | 92.9% | 90.1% |

### XAI Method Comparison

| Property | SHAP | LIME |
|---|---|---|
| **Scope** | Global + Local | Local (aggregated globally) |
| **Theoretical basis** | Shapley values (game theory) | Local linear approximation |
| **Best use** | Model-level climate driver analysis | Field-level explanation to farmer |
| **SDG relevance** | SDG 13 — confirms climate variable dominance | SDG 2 — farmer-interpretable decisions |

### 🌱 Key Sustainability Findings (MCC471)

| Finding | Environmental Significance |
|---|---|
| **Climate features (Rainfall + Temperature)** dominate predictions | Directly validates SDG 13 — climate is the primary crop suitability driver |
| **Potassium (K)** is the top soil driver | Guides precision fertilisation → SDG 12 & SDG 15 |
| **Drought-tolerant crops** cluster at <100mm rainfall | Tool can guide climate migration of crops under changing rainfall → SDG 13 |
| **SHAP–LIME agreement >0.85** | Robust, trustworthy insights — safe for policy recommendations |

### Why Decision Tree is Selected for Deployment
- Highest **precision** (95.56%) — fewest wrong recommendations
- Human-readable decision rules → explainable to farmers without ML background (SDG 2)
- Fast inference → suitable for mobile/edge deployment in rural areas
- SHAP tree-path values directly correspond to split contributions

---
## 🌾 12. Climate-Adaptive Crop Advisory Tool (Live Predictor)

> **Practical deliverable for SDG 2 & SDG 13:**  
> Given real-time soil test results and local weather data, this tool recommends the optimal crop  
> with a LIME explanation — translating environmental data into actionable, sustainable farming decisions.

In [ ]:
def recommend_crop(N, P, K, pH, rainfall, temperature, top_k=3):
    """
    Climate-Adaptive Crop Advisory Tool (MCC471 — SDG 2, 6, 12, 13, 15)

    Parameters
    ----------
    N, P, K     : Soil nutrient levels (mg/kg)   [SDG 12 / SDG 15]
    pH          : Soil pH                          [SDG 15]
    rainfall    : Local annual rainfall (mm)       [SDG 6 / SDG 13]
    temperature : Mean annual temperature (°C)     [SDG 13]
    top_k       : Number of alternative crops to show
    """
    x        = np.array([[N, P, K, pH, rainfall, temperature]])
    rf_model = trained_models['Random Forest']
    probs    = rf_model.predict_proba(x)[0]
    top_idx  = np.argsort(probs)[::-1][:top_k]

    print(f'\n{"═"*60}')
    print(f'  🌱 CLIMATE-ADAPTIVE CROP ADVISORY TOOL')
    print(f'  Course: MCC471 — Sustainability & Climate Sciences')
    print(f'{'─'*60}')
    print(f'  🌿 Soil    : N={N} mg/kg  P={P} mg/kg  K={K} mg/kg  pH={pH}')
    print(f'  🌡️  Climate : Rainfall={rainfall} mm   Temperature={temperature}°C')
    print(f'{'─'*60}')
    print(f'  📋 Top {top_k} Crop Recommendations:')
    for rank, idx in enumerate(top_idx, 1):
        filled = int(probs[idx] * 30)
        bar    = '█'*filled + '░'*(30-filled)
        print(f'  #{rank}  {le.classes_[idx]:<20s}  {bar}  {probs[idx]:.1%}')
    print(f'{'═'*60}')

    # LIME explanation for top crop
    rng_l  = np.random.default_rng(42)
    std    = X_train.std(0) + 1e-8
    noise  = rng_l.normal(0, 1, (800, len(FEATURES)))
    X_pert = np.clip(x[0] + noise*std*0.4, X_train.min(0), X_train.max(0))
    pc     = top_idx[0]
    pr     = rf_model.predict_proba(X_pert)[:, pc]
    dist   = np.sqrt(((X_pert - x[0])**2 / std**2).sum(1))
    w      = np.exp(-(dist**2)/(2*(np.sqrt(len(FEATURES))*0.3)**2))
    X_n    = (X_pert - X_pert.mean(0))/(X_pert.std(0)+1e-8)
    reg    = Ridge(alpha=1.0)
    reg.fit(X_n, pr, sample_weight=w)
    coef   = reg.coef_

    fig, ax = plt.subplots(figsize=(10, 5))
    order  = np.argsort(np.abs(coef))
    vals   = coef[order]
    bar_labels = []
    for i in order:
        icon = '🌡️' if FEATURES[i] in CLIMATE_FEATURES else '🌿'
        bar_labels.append(f'{icon} {FLABELS[i]}  ({x[0,i]:.2f})')
    cols = ['#E74C3C' if v > 0 else '#3498DB' for v in vals]
    ax.barh(bar_labels, vals, color=cols, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', lw=1.2, alpha=0.7)
    ax.set_title(
        f'🌾 LIME Explanation — Why {le.classes_[pc].upper()} is Recommended\n'
        f'Confidence: {probs[pc]:.1%}  |  🌡️ Climate (SDG 13)  🌿 Soil (SDG 15)',
        fontweight='bold', fontsize=12
    )
    ax.set_xlabel('LIME Coefficient — Local Climate/Soil Influence on Recommendation')
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()


# ── Test scenarios aligned with SDG context ─────────────────────────────
print('\n🔴 Scenario 1: Rice-growing region (high rainfall, warm)')
recommend_crop(N=80, P=40, K=40, pH=5.66, rainfall=297.66, temperature=29.57)

print('\n🟡 Scenario 2: Barley-suited semi-arid region (low rainfall, mild)')
recommend_crop(N=70, P=40, K=45, pH=5.54, rainfall=75.32,  temperature=22.68)

print('\n🟢 Scenario 3: Climate-stressed region (low water, cool)')
recommend_crop(N=20, P=30, K=20, pH=7.00, rainfall=55.0,   temperature=18.0)

---
## ⚠️ 13. Challenges & Limitations

| Challenge | Detail |
|---|---|
| **Static climate data** | Dataset uses historical averages; real deployment needs live IoT/weather feeds |
| **No temporal dimension** | Cannot model seasonal shifts or year-on-year climate trends (LSTM needed) |
| **Regional specificity** | Dataset represents Indian agro-climatic zones; not globally generalisable |
| **Missing variables** | No pest risk, market price, water table, or soil moisture data |
| **Custom XAI approximations** | SHAP tree-path and LIME batch are custom implementations — not the official libraries |
| **Rural deployment** | Mobile/edge infrastructure needed for smallholder farmer access |

---
## 🚀 14. Future Scope

| Extension | SDG Impact |
|---|---|
| Integrate IoT soil sensors + real-time weather APIs | SDG 13 — live climate adaptation |
| Add **carbon footprint score** per crop recommendation | SDG 13 — emissions-aware farming |
| Include **water consumption estimate** per crop | SDG 6 — precision water management |
| **LSTM forecasting** for seasonal climate trends | SDG 13 — forward-looking recommendations |
| Mobile app deployment in regional languages | SDG 2 — inclusive farmer access |
| District/state-level policy recommendations with GIS | SDG 11, 13 — urban-rural sustainability planning |
| Extend to global agro-climatic zones (Africa, SE Asia) | SDG 2 — food security at scale |